In [1]:
from pathlib import Path
import zipfile
import tempfile
import pandas as pd

zip_path = Path("/content/week6_feature_engineering_output.zip")

if not zip_path.is_file():
    raise FileNotFoundError(f"Please upload {zip_path.name}")

data_folder = Path(
    tempfile.mkdtemp(prefix="week7_inputs_", dir="/content")
)

with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(data_folder)

def load_engineered_csv(filename):
    matches = list(data_folder.rglob(filename))

    if len(matches) != 1:
        raise ValueError(
            f"Expected one {filename}; found {len(matches)}."
        )

    return pd.read_csv(
        matches[0],
        dtype={"ListingKey": "string", "PostalCode": "string"},
        low_memory=False
    )

sales = load_engineered_csv("train_engineered.csv")
september = load_engineered_csv("september_engineered.csv")

dates = pd.to_datetime(sales["CloseDate"], errors="raise", utc=True)

train = sales.loc[
    (dates >= "2024-09-01") & (dates < "2026-06-01")
].copy()

validation = sales.loc[
    (dates >= "2026-06-01") & (dates < "2026-09-01")
].copy()

print(f"Training sales: {len(train):,}")
print(f"June–August validation sales: {len(validation):,}")
print(f"September test sales: {len(september):,}")
print(f"Maximum training sale price: ${train['ClosePrice'].max():,.0f}")

Training sales: 227,502
June–August validation sales: 37,781
September test sales: 11,186
Maximum training sale price: $17,000,000


In [2]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_columns = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt"
]

engineered_columns = [
    "bed_bath_ratio",
    "property_age_years"
]

categorical_columns = [
    "City",
    "CountyOrParish",
    "PostalCode",
    "school_district_elementary",
    "school_district_high",
    "school_district_unified"
]

flag_columns = [
    f"{column}_{suffix}"
    for column in numeric_columns
    for suffix in ["invalid_flag", "missing_flag"]
]

feature_columns = (
    numeric_columns
    + engineered_columns
    + categorical_columns
    + flag_columns
)

def prepare_features(data):
    features = data[feature_columns].copy()

    for flag in flag_columns:
        features[flag] = (
            features[flag].astype("string")
            .str.strip().str.lower()
            .isin(["true", "1", "1.0"])
            .astype("int8")
        )

    # Restore missing predictors previously filled in Week 3.
    for column in numeric_columns:
        features[column] = pd.to_numeric(
            features[column], errors="coerce"
        )
        features.loc[
            features[f"{column}_missing_flag"].eq(1),
            column
        ] = np.nan

    # Recalculate engineered values using the restored predictors.
    bathrooms = features["BathroomsTotalInteger"].where(
        features["BathroomsTotalInteger"] > 0
    )
    features["bed_bath_ratio"] = (
        features["BedroomsTotal"] / bathrooms
    )

    sale_year = pd.to_datetime(
        data["CloseDate"], errors="raise", utc=True
    ).dt.year

    age = sale_year - features["YearBuilt"]
    features["property_age_years"] = age.where(age >= 0)

    features[numeric_columns + engineered_columns] = (
        features[numeric_columns + engineered_columns]
        .replace([np.inf, -np.inf], np.nan)
    )

    for column in categorical_columns:
        features[column] = (
            features[column].astype("string")
            .str.strip().replace("", pd.NA)
            .fillna("Unknown").astype(str)
        )

    return features


X_train_raw = prepare_features(train)
X_validation_raw = prepare_features(validation)

y_train = train["ClosePrice"].astype(float)
y_validation = validation["ClosePrice"].astype(float)

preprocessor = ColumnTransformer([
    (
        "numeric",
        SimpleImputer(strategy="median"),
        numeric_columns + engineered_columns
    ),
    (
        "categorical",
        OneHotEncoder(
            handle_unknown="ignore",
            dtype=np.float32
        ),
        categorical_columns
    ),
    ("flags", "passthrough", flag_columns)
], sparse_threshold=1.0)

X_train = preprocessor.fit_transform(X_train_raw)
X_validation = preprocessor.transform(X_validation_raw)

assert X_train.shape[0] == len(y_train)
assert X_validation.shape[0] == len(y_validation)
assert X_train.shape[1] == X_validation.shape[1]

print("Training feature shape:", X_train.shape)
print("Validation feature shape:", X_validation.shape)
print("Features prepared. Ready for XGBoost training.")

Training feature shape: (227502, 4632)
Validation feature shape: (37781, 4632)
Features prepared. Ready for XGBoost training.


In [3]:
from pathlib import Path
from time import perf_counter
from xgboost import XGBRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

configurations = [
    {"max_depth": 4, "learning_rate": 0.10, "n_estimators": 200},
    {"max_depth": 6, "learning_rate": 0.10, "n_estimators": 200},
    {"max_depth": 6, "learning_rate": 0.05, "n_estimators": 400},
    {"max_depth": 6, "learning_rate": 0.10, "n_estimators": 400}
]

output_folder = Path("/content/week7_advanced_models_output")
output_folder.mkdir(parents=True, exist_ok=True)

tuning_results = []
best_validation_rmse = float("inf")
best_model = None
best_parameters = None

for run_number, parameters in enumerate(configurations, start=1):
    print(
        f"\nStarting run {run_number}/{len(configurations)}: {parameters}",
        flush=True
    )

    candidate = XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        random_state=42,
        n_jobs=2,
        **parameters
    )

    start = perf_counter()
    candidate.fit(X_train, y_train)
    fit_seconds = perf_counter() - start

    predicted = candidate.predict(X_validation)

    validation_rmse = np.sqrt(
        mean_squared_error(y_validation, predicted)
    )

    tuning_results.append({
        **parameters,
        "Validation_R2": r2_score(y_validation, predicted),
        "Validation_MAE": mean_absolute_error(
            y_validation, predicted
        ),
        "Validation_RMSE": validation_rmse,
        "Fit_seconds": fit_seconds
    })

    if validation_rmse < best_validation_rmse:
        best_validation_rmse = validation_rmse
        best_model = candidate
        best_parameters = parameters.copy()

    # Save progress after each completed run.
    pd.DataFrame(tuning_results).to_csv(
        output_folder / "xgboost_validation_results.csv",
        index=False
    )

    print(
        f"Finished in {fit_seconds:.1f} seconds. "
        f"Validation RMSE: ${validation_rmse:,.0f}",
        flush=True
    )

tuning_table = (
    pd.DataFrame(tuning_results)
    .sort_values("Validation_RMSE")
)

display(tuning_table.style.format({
    "Validation_R2": "{:.4f}",
    "Validation_MAE": "${:,.0f}",
    "Validation_RMSE": "${:,.0f}",
    "Fit_seconds": "{:.1f}"
}))

print("\nSelected parameters:", best_parameters)



Starting run 1/4: {'max_depth': 4, 'learning_rate': 0.1, 'n_estimators': 200}
Finished in 10.4 seconds. Validation RMSE: $566,841

Starting run 2/4: {'max_depth': 6, 'learning_rate': 0.1, 'n_estimators': 200}
Finished in 6.8 seconds. Validation RMSE: $528,871

Starting run 3/4: {'max_depth': 6, 'learning_rate': 0.05, 'n_estimators': 400}
Finished in 15.2 seconds. Validation RMSE: $532,391

Starting run 4/4: {'max_depth': 6, 'learning_rate': 0.1, 'n_estimators': 400}
Finished in 17.0 seconds. Validation RMSE: $495,616


,max_depth,learning_rate,n_estimators,Validation_R2,Validation_MAE,Validation_RMSE,Fit_seconds
3,6,0.100000,400,0.8299,"$231,757","$495,616",17.0
1,6,0.100000,200,0.8064,"$259,948","$528,871",6.8
2,6,0.050000,400,0.8038,"$261,593","$532,391",15.2
0,4,0.100000,200,0.7776,"$288,812","$566,841",10.4



Selected parameters: {'max_depth': 6, 'learning_rate': 0.1, 'n_estimators': 400}


The XGBoost configuration with depth 6, a learning rate of 0.1, and 400 trees performed best on validation

It had the lowest validation RMSE ($495,616) and    MAE ($231,757), with R²   of    0.8299

In [4]:
from sklearn.base import clone

# Prepare all September 2024–August 2026 training sales.
X_final_raw = prepare_features(sales)
X_test_raw = prepare_features(september)

y_final = sales["ClosePrice"].astype(float)
y_test = september["ClosePrice"].astype(float)

assert y_final.le(17_000_000).all()

# Relearn medians and categories using all final training months.
final_preprocessor = clone(preprocessor)

X_final = final_preprocessor.fit_transform(X_final_raw)
X_test = final_preprocessor.transform(X_test_raw)

final_xgboost = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=2,
    **best_parameters
)

start = perf_counter()
final_xgboost.fit(X_final, y_final)
final_fit_seconds = perf_counter() - start

test_predictions = final_xgboost.predict(X_test)

test_results = pd.DataFrame([{
    "Model": "XGBoost",
    **best_parameters,
    "Training_rows": len(y_final),
    "September_rows": len(y_test),
    "September_R2": r2_score(y_test, test_predictions),
    "September_MAE": mean_absolute_error(y_test, test_predictions),
    "September_RMSE": np.sqrt(
        mean_squared_error(y_test, test_predictions)
    ),
    "Negative_predictions": int((test_predictions < 0).sum()),
    "Fit_seconds": final_fit_seconds
}])

test_results.to_csv(
    output_folder / "xgboost_september_results.csv",
    index=False
)

display(test_results.style.format({
    "September_R2": "{:.4f}",
    "September_MAE": "${:,.0f}",
    "September_RMSE": "${:,.0f}",
    "Fit_seconds": "{:.1f}"
}))

,Model,max_depth,learning_rate,n_estimators,Training_rows,September_rows,September_R2,September_MAE,September_RMSE,Negative_predictions,Fit_seconds
0,XGBoost,6,0.100000,400,265283,11186,0.7751,"$252,736","$674,391",0,16.8


XGBoost reduced average absolute error by $46,322 per sale (15.5%) and increased R² by 4.64 percentage points. Both models used the same training and September sales with the property and district features.

In [5]:
import joblib
import shutil
from google.colab import files

final_xgboost.save_model(
    str(output_folder / "xgboost_model.json")
)

joblib.dump(
    {
        "preprocessor": final_preprocessor,
        "feature_columns": feature_columns,
        "numeric_columns": numeric_columns,
        "engineered_columns": engineered_columns,
        "categorical_columns": categorical_columns,
        "flag_columns": flag_columns
    },
    output_folder / "xgboost_preprocessing.joblib"
)

pd.DataFrame({
    "Actual_ClosePrice": y_test.to_numpy(),
    "Predicted_ClosePrice": test_predictions
}).to_csv(
    output_folder / "september_predictions.csv",
    index=False
)

zip_path = shutil.make_archive(
    str(output_folder), "zip", str(output_folder)
)

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Week 7 — XGBoost Results

Four XGBoost configurations were compared using June–August 2026 validation data.

**Selected settings**

- Maximum depth: 6
- Learning rate: 0.1
- Number of trees: 400

The selected model was trained on 265,283 sales and evaluated on all 11,186 September 2026 sales.

| September metric | Result |
|---|---:|
| R² | 0.7751 |
| MAE | USD 252,736 |
| RMSE | USD 674,391 |
| Negative predictions | 0 |

XGBoost achieved better September results than the Week 6 models.

**Evaluation notes**

- Training and validation included sales priced at 17 million dollars or less.
- September evaluation included all test sales, including those above that cutoff.
